# Experiment 1 — Fabry–Pérot interferometer at São João do Cariri

**Question:** How accurately can an FPI recover thermospheric winds and temperature from OI 630.0 nm emission? This standalone OSSE prescribes altitude-dependent winds and temperature, weights the emission through a configurable layer, forms a Doppler/thermal line, applies an Airy etalon response, simulates detector counts, and retrieves line-of-sight winds. Site: **7.4° S, 36.5° W**; North/East looks are at 45° elevation and zenith is an optional vertical-wind reference.

## Equations, units, assumptions

\(v_{LOS}=\cos(e)[u\sin A+v\cos A]+w\sin(e)\), with azimuth clockwise from north and winds in m s⁻¹. The emitting layer is Gaussian in altitude. Oxygen thermal width is \(\sigma_v=\sqrt{k_BT/m_O}\). The Airy response is \(T=[1+F\sin^2(2\pi nd\cos\alpha/\lambda)]^{-1}\), with \(F=(2\mathcal{F}/\pi)^2\). Detector noise is Poisson shot noise plus dark current and Gaussian read noise.

Reference starting points are a 1.5 cm gap, 7–10 cm aperture, and 630.0 nm line. QE, exposure, brightness scale, throughput, finesse, focal length, and sampling are illustrative, not verified specifications for a Cariri instrument. A finite layer returns an emission-weighted average, not wind at one exact altitude. North/east alone cannot separate vertical wind; the main retrieval also uses zenith, while the two-look fallback assumes w=0.

In [1]:
from pathlib import Path
import os
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy.optimize import least_squares
from scipy.linalg import pinv
SEED=6302026
ROOT=Path(os.environ.get("SBGEA_KEYNOTE_DATA",Path.home()/"Documents/Science/Programs/LFS/KEYNOTE")).expanduser()
OUT=ROOT/"outputs"; FIG=OUT/"figures"/"keynote"; DATA=OUT/"data"/"osse"
FIG.mkdir(parents=True,exist_ok=True); DATA.mkdir(parents=True,exist_ok=True)
I={"lam":630.0304e-9,"gap":.015,"aperture":.08,"finesse":22.,"qe":.65,"exposure":180.,"throughput":.22,"rate":140.,"dark":.04,"read":4.,"instr":35.,"n":19,"field":.020}
peak=250.; fwhm=60.; Z=np.linspace(150,350,401); C0=299792458.; KB=1.380649e-23; MO=16*1.66053906660e-27
Gx,Gw=np.polynomial.hermite.hermgauss(9); Gw=Gw/np.sqrt(np.pi)
xx,yy=np.meshgrid(np.linspace(-1,1,I["n"]),np.linspace(-1,1,I["n"])); theta=I["field"]*np.hypot(xx,yy)
looks={"Norte · 45°":(0.,45.),"Leste · 45°":(90.,45.),"Zênite":(0.,90.)}
def ue(z):
 d=np.asarray(z)-peak; return 70+.10*d+.004*d*d
def vn(z):
 d=np.asarray(z)-peak; return -20+.06*d-.002*d*d
def wu(z):
 d=np.asarray(z)-peak; return 6+.025*d
def temp(z):
 d=np.asarray(z)-peak; return 850+.25*d+.004*d*d
def layer(f):
 s=f/(2*np.sqrt(2*np.log(2))); q=np.exp(-.5*((Z-peak)/s)**2); return q/np.trapezoid(q,Z)
def coeff(az,el):
 a,e=np.deg2rad([az,el]); return np.array([np.cos(e)*np.sin(a),np.cos(e)*np.cos(a),np.sin(e)])
def moments(name,f):
 w=layer(f); c=coeff(*looks[name]); vl=c@np.vstack([ue(Z),vn(Z),wu(Z)])
 mu=np.trapezoid(w*vl,Z); T=np.trapezoid(w*temp(Z),Z); shear=np.sqrt(np.trapezoid(w*(vl-mu)**2,Z))
 return float(mu),float(T),float(shear),float(np.trapezoid(w*ue(Z),Z)),float(np.trapezoid(w*vn(Z),Z))
def shape(v,T,inst=I,shear=0.):
 sig=np.sqrt(KB*T/MO+inst["instr"]**2+shear**2); lam=inst["lam"]*(1+(v+np.sqrt(2)*sig*Gx)/C0)
 phase=4*np.pi*inst["gap"]*np.cos(theta)[...,None]/lam
 F=(2*inst["finesse"]/np.pi)**2; a=1/(1+F*np.sin(phase/2)**2); im=np.sum(a*Gw,axis=-1); return im/im.mean()
def expected(v,T,inst=I,amp=None,shear=0.):
 if amp is None: amp=inst["rate"]*inst["exposure"]*inst["qe"]*inst["throughput"]
 return 28+inst["dark"]*inst["exposure"]+amp*shape(v,T,inst,shear)
def observe(v,T,seed,inst=I,shear=0.):
 r=np.random.default_rng(seed); mu=expected(v,T,inst,shear=shear); return np.maximum(r.poisson(mu)+r.normal(0,inst["read"],mu.shape),0),mu
def fit(y,inst=I,init=(0.,850.)):
 y=np.asarray(y); sig=np.sqrt(np.maximum(y+inst["read"]**2,1))
 def residual(p):
  q=shape(p[0],p[1],inst); B=np.column_stack([q.ravel(),np.ones(q.size)]); ab=np.linalg.lstsq(B,y.ravel(),rcond=None)[0]
  return ((y-ab[0]*q-ab[1])/sig).ravel()
 r=least_squares(residual,init,bounds=([-350,400],[350,1800]),max_nfev=45)
 return {"v":float(r.x[0]),"T":float(r.x[1]),"success":bool(r.success)}
def fisher(mu,T,inst=I,shear=0.):
 amp=inst["rate"]*inst["exposure"]*inst["qe"]*inst["throughput"]; q=shape(mu,T,inst,shear)
 dv=.5; dT=2.; qv=(shape(mu+dv,T,inst,shear)-shape(mu-dv,T,inst,shear))/(2*dv); qt=(shape(mu,T+dT,inst,shear)-shape(mu,T-dT,inst,shear))/(2*dT)
 m=28+inst["dark"]*inst["exposure"]+amp*q; var=m+inst["read"]**2
 J=np.column_stack([amp*qv.ravel(),amp*qt.ravel(),q.ravel(),np.ones(q.size)]); cov=pinv(J.T@(J/var.ravel()[:,None])); return float(np.sqrt(max(cov[0,0],0)))

In [2]:
# Forward simulation, fit, geometry combination, Monte Carlo, and sensitivity.
truth={}; images={}; fits={}
for j,k in enumerate(looks):
 m,t,s,_,_=moments(k,fwhm); images[k],_=observe(m,t,SEED+10+j,shear=s); truth[k]=(m,t,s); fits[k]=fit(images[k],init=(m,t))
H=np.vstack([coeff(*looks[k]) for k in looks]); y=np.array([fits[k]["v"] for k in looks]); sig=np.array([fisher(*truth[k][:2],shear=truth[k][2]) for k in looks]); W=np.diag(1/sig**2); cov=np.linalg.inv(H.T@W@H); estimate=cov@H.T@W@y
truth_peak=np.array([ue(peak),vn(peak),wu(peak)])
print("Retrieved east/north/up (m/s):",np.round(estimate,2)," truth:",truth_peak)
print("Formal 1σ (m/s):",np.round(np.sqrt(np.diag(cov)),2)," fits converged:",all(x["success"] for x in fits.values()))
mc=[]
for r in range(10):
 vals=[]
 for j,k in enumerate(looks):
  m,t,s=truth[k]; obs,_=observe(m,t,SEED+1000+10*r+j,shear=s); vals.append(fit(obs,init=(m,t))["v"])
 mc.append(np.linalg.solve(H.T@W@H,H.T@W@vals))
mc=np.asarray(mc); mc_sigma=mc.std(axis=0,ddof=1); print("Repeatability σ east/north/up (m/s):",np.round(mc_sigma,2))
qe_grid=np.array([.20,.40,.60,.80,.95]); fgrid=np.array([20.,40.,60.,80.,100.]); records=[]; formal=np.zeros((5,5)); bias=formal.copy(); total=formal.copy()
for i,f in enumerate(fgrid):
 mm={k:moments(k,f) for k in looks}; b=np.hypot(mm["Norte · 45°"][3]-ue(peak),mm["Leste · 45°"][4]-vn(peak))
 for j,qe in enumerate(qe_grid):
  inst={**I,"qe":qe}; ss=np.array([fisher(*mm[k][:2],inst,shear=mm[k][2]) for k in looks]); cv=np.linalg.inv(H.T@np.diag(1/ss**2)@H)
  formal[i,j]=np.sqrt(np.trace(cv[:2,:2])); bias[i,j]=b; total[i,j]=np.hypot(formal[i,j],b)
  records.append({"fwhm_km":f,"qe":qe,"formal_horizontal_sigma_m_s":formal[i,j],"representativeness_bias_m_s":b,"total_error_estimate_m_s":total[i,j]})
sens=pd.DataFrame(records); sens.to_csv(DATA/"experiment_1_sensitivity_reconstructed.csv",index=False)
assert formal[2,-1]<formal[2,0] and bias[-1,0]>bias[0,0]
assert all(x["success"] for x in fits.values()) and np.all(np.isfinite(estimate))
np.savez_compressed(DATA/"experiment_1_payload_reconstructed.npz",qe=qe_grid,fwhm_km=fgrid,total_error=total,formal_error=formal,repr_bias=bias,image=images["Norte · 45°"],north_image=shape(*truth["Norte · 45°"][:2],shear=truth["Norte · 45°"][2]),east_image=shape(*truth["Leste · 45°"][:2],shear=truth["Leste · 45°"][2]),z_km=Z,east_wind=ue(Z),north_wind=vn(Z),mc_sigma=mc_sigma,truth_east=ue(peak),truth_north=vn(peak))

Retrieved east/north/up (m/s): [ 72.36 -21.77   6.23]  truth: [ 70. -20.   6.]
Formal 1σ (m/s): [0.48 0.57 0.34]  fits converged: True
Repeatability σ east/north/up (m/s): [0.43 0.8  0.29]


In [3]:
# Projection-ready 11.25 × 5.5 in slide; all panels use this cell's numerical results.
fig=plt.figure(figsize=(11.25,5.5),dpi=300,facecolor="white"); gs=fig.add_gridspec(2,3,left=.055,right=.97,top=.82,bottom=.20,wspace=.38,hspace=.42,height_ratios=[1,.7])
ax0=fig.add_subplot(gs[0,0]); ax1=fig.add_subplot(gs[0,1]); ax2=fig.add_subplot(gs[:,2]); ax3=fig.add_subplot(gs[1,:2])
fig.suptitle("INTERFEROMETRIA · São João do Cariri",x=.055,ha="left",fontsize=18,fontweight="bold",color="#082F73")
fig.text(.055,.87,"Como a eficiência do detector afeta a precisão dos ventos?",fontsize=11,color="#25364A")
im=ax0.imshow(images["Norte · 45°"],origin="lower",cmap="viridis"); ax0.set_title("Interferograma sintético",loc="left",fontsize=10,color="#082F73"); ax0.axis("off"); fig.colorbar(im,ax=ax0,fraction=.05,pad=.03,label="contagens")
ax1.set(xlim=(-1.2,1.3),ylim=(-.15,1.6)); ax1.set_aspect("equal"); ax1.axis("off"); ax1.scatter([0],[0],s=55,color="#082F73")
for label,xy,col in [("Norte",(0,1),"#2686D0"),("Leste",(.72,.72),"#EC8135"),("Zênite",(0,1.45),"#70AE54")]: ax1.annotate("",xy=xy,xytext=(0,0),arrowprops=dict(arrowstyle="->",lw=2,color=col)); ax1.text(xy[0]+.05,xy[1],label,color=col,fontsize=8)
ax1.set_title("Geometria de observação",loc="left",fontsize=10,color="#082F73")
mesh=ax2.pcolormesh(qe_grid*100,fgrid,total,shading="nearest",cmap="magma"); ax2.set(xlabel="Eficiência quântica (%)",ylabel="FWHM da camada (km)"); ax2.set_title("Erro total estimado (m/s)",loc="left",fontsize=10,color="#082F73"); fig.colorbar(mesh,ax=ax2,fraction=.05,pad=.04,label="m/s")
ax3.plot(fgrid,formal[:,2],"o-",color="#2686D0",label="ruído · QE 60%"); ax3.plot(fgrid,bias[:,2],"s--",color="#EC8135",label="viés de representatividade"); ax3.set(xlabel="FWHM da camada emissora (km)",ylabel="Erro horizontal (m/s)"); ax3.grid(alpha=.2); ax3.legend(frameon=False,ncol=2,fontsize=7)
fig.text(.055,.105,f"Caso nominal: σ formal horizontal = {np.sqrt(np.trace(cov[:2,:2])):.1f} m/s; viés da camada = {bias[2,2]:.1f} m/s.",fontsize=8.5,color="#25364A")
fig.text(.055,.055,"Mais fótons reduzem ruído; não removem o viés de média vertical. QE e brilho são hipóteses ilustrativas.",fontsize=7.5,color="#63758A")
fig.savefig(FIG/"Experiment_1_Slide.png",facecolor="white"); fig.savefig(FIG/"Experiment_1_Slide_Recovered.png",facecolor="white"); plt.show()
fig2,ax=plt.subplots(figsize=(7,4));
for i,f in enumerate(fgrid): ax.plot(qe_grid*100,formal[i],"o-",label=f"FWHM {f:.0f} km")
ax.set(xlabel="Eficiência quântica (%)",ylabel="σ formal horizontal (m/s)",title="Sensibilidade à eficiência do CCD"); ax.grid(alpha=.2); ax.legend(frameon=False,ncol=2); fig2.tight_layout(); fig2.savefig(FIG/"Experiment_1_Sensitivity_Recovered.png",dpi=180); plt.show()
# Additional independent sensitivities: exposure, emission brightness, throughput, and instrumental width.
extra=[]
for parameter,values in {"exposure_s":[30.,60.,120.,180.,300.],"emission_rate":[40.,70.,140.,280.],"throughput":[.08,.15,.22,.40],"instrument_width_m_s":[15.,35.,70.]}.items():
 for value in values:
  inst={**I}
  if parameter=="exposure_s": inst["exposure"]=value
  elif parameter=="emission_rate": inst["rate"]=value
  elif parameter=="throughput": inst["throughput"]=value
  else: inst["instr"]=value
  mm={k:moments(k,60.) for k in ("Norte · 45°","Leste · 45°","Zênite")}
  ss=np.array([fisher(*mm[k][:2],inst,shear=mm[k][2]) for k in mm])
  cv=np.linalg.inv(H.T@np.diag(1/ss**2)@H)
  extra.append({"parameter":parameter,"value":value,"horizontal_formal_sigma_m_s":np.sqrt(np.trace(cv[:2,:2]))})
pd.DataFrame(extra).to_csv(DATA/"experiment_1_instrument_sensitivity_reconstructed.csv",index=False)
print("Outputs:",DATA/"experiment_1_payload_reconstructed.npz",DATA/"experiment_1_sensitivity_reconstructed.csv",FIG/"Experiment_1_Slide_Recovered.png")

Outputs: $SBGEA_KEYNOTE_DATA/outputs/data/osse/experiment_1_payload_reconstructed.npz $SBGEA_KEYNOTE_DATA/outputs/data/osse/experiment_1_sensitivity_reconstructed.csv $SBGEA_KEYNOTE_DATA/outputs/figures/keynote/Experiment_1_Slide_Recovered.png


/var/folders/6m/4lrc3ql56_g0hsgdbnj5t18r0000gn/T/ipykernel_80226/1865236528.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.savefig(FIG/"Experiment_1_Slide.png",facecolor="white"); fig.savefig(FIG/"Experiment_1_Slide_Recovered.png",facecolor="white"); plt.show()
/var/folders/6m/4lrc3ql56_g0hsgdbnj5t18r0000gn/T/ipykernel_80226/1865236528.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  ax.set(xlabel="Eficiência quântica (%)",ylabel="σ formal horizontal (m/s)",title="Sensibilidade à eficiência do CCD"); ax.grid(alpha=.2); ax.legend(frameon=False,ncol=2); fig2.tight_layout(); fig2.savefig(FIG/"Experiment_1_Sensitivity_Recovered.png",dpi=180); plt.show()


## Limitations, references, and student extensions

The instrument image uses a monochromatic ideal Airy response and a Gaussian effective line; it omits calibration drift, etalon defects, sky background structure, spectral contamination, and detailed CCD calibration. Measurement noise and vertical representativeness bias are reported separately. Vertical wind is retrieved only because the zenith look is included; without it the horizontal inversion assumes w=0.

Starting references: [DLR FPI instrument overview](https://www.dlr.de/en/so/research-transfer/research-infrastructure/instruments/fabry-perot-interferometer), [JGR Space Physics (2014)](https://doi.org/10.1002/2014JA019832), [Annales Geophysicae (2018)](https://angeo.copernicus.org/articles/36/265/2018/).

### What could an undergraduate investigate next?
1. How do exposure time and sky background affect precision?
2. Which look geometry best separates east, north, and vertical winds?
3. How does a multi-layer or non-Gaussian emission profile change the retrieval bias?

### Recovery provenance

This executed notebook is a reconstructed, reproducible implementation informed by the project specification and preserved OSSE tables/payloads. The original notebook bytes were deleted during migration. Their original SHA-256 values remain in `KEYNOTE/MIGRATION_MANIFEST.csv`; the rebuilt source is not represented as a byte-identical recovery.

## Interactive FPI experiment design

Adjust detector efficiency, exposure, and emission-layer thickness. The cell regenerates the synthetic interferogram, refits the Doppler/thermal line, and reports the formal line-of-sight uncertainty. This is an instrument-noise/representativeness exploration, not a verified Cariri instrument forecast.

In [4]:
import ipywidgets as widgets
from IPython.display import display, clear_output
fpi_out=widgets.Output()
def _fpi_explore(qe=65, exposure_s=180, layer_fwhm_km=60):
    with fpi_out:
        clear_output(wait=True)
        inst={**I,"qe":qe/100.0,"exposure":float(exposure_s)}
        v,T,shear,_,_=moments("Norte · 45°",float(layer_fwhm_km))
        image,_=observe(v,T,SEED+440,shear=shear,inst=inst)
        recovered=fit(image,inst=inst,init=(v,T))
        sigma=fisher(v,T,inst,shear=shear)
        fig,ax=plt.subplots(1,2,figsize=(8,3.1),facecolor="white")
        ax[0].imshow(image,origin="lower",cmap="viridis"); ax[0].set_title("Interferograma sintético")
        ax[0].set_axis_off()
        ax[1].bar(["vento LOS","temperatura"],[recovered["v"],recovered["T"]],color=["#2686D0","#EC8135"])
        ax[1].set_ylabel("m/s e K (escalas distintas)"); ax[1].set_title("Ajuste do perfil espectral"); ax[1].grid(axis="y",alpha=.2)
        fig.suptitle(f"σ formal vento = {sigma:.2f} m/s | camada FWHM = {layer_fwhm_km} km")
        fig.tight_layout(); plt.show()
        print(f"Truth LOS={v:.2f} m/s; fit={recovered['v']:.2f} m/s; T={recovered['T']:.1f} K; converged={recovered['success']}")
controls=widgets.interactive(_fpi_explore,
    qe=widgets.IntSlider(value=65,min=20,max=95,step=5,description="QE (%)"),
    exposure_s=widgets.IntSlider(value=180,min=30,max=300,step=30,description="Expo. (s)"),
    layer_fwhm_km=widgets.IntSlider(value=60,min=20,max=100,step=10,description="Camada (km)"))
display(controls)


interactive(children=(IntSlider(value=65, description='QE (%)', max=95, min=20, step=5), IntSlider(value=180, …